# 因子测试 · 03：融合因子与 implication 单因子回归

用 **fusion** 对三个 implication 变量分别做**单因子回归**（因变量=fusion，每次一个自变量）：
- fusion ~ 大盘涨跌-小盘涨跌
- fusion ~ 中国波动率收盘价
- fusion ~ 中证股债风险平均指数收盘价

依赖：01_load_y_and_implication.xlsx、04_fusion_timeseries.xlsx

In [9]:
import os
import sys
import pandas as pd
import numpy as np

_cwd = os.getcwd()
_root = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'factor_test' else _cwd
if _root not in sys.path:
    sys.path.insert(0, _root)
import config

DATE_COL = config.DATE_COL
FB_OUT = os.path.join(_root, 'factor_build', getattr(config, 'FACTOR_BUILD_OUTPUTS', 'outputs'))
FT_OUT = os.path.join(_root, 'factor_test', getattr(config, 'FACTOR_TEST_OUTPUTS', 'outputs'))
Y_COL = 'fusion'  # 因变量：融合因子
X_NAMES = ['大盘涨跌-小盘涨跌', '中国波动率收盘价', '中证股债风险平均指数收盘价']  # 自变量

## 1. 读取 01 输出与融合因子

In [10]:
path_impl = os.path.join(FT_OUT, '01_y_and_implication.xlsx')
path_fusion = os.path.join(FB_OUT, '04_fusion_timeseries.xlsx')
if not os.path.isfile(path_impl):
    raise FileNotFoundError('请先运行 factor_test/01，生成 01_y_and_implication.xlsx')
if not os.path.isfile(path_fusion):
    raise FileNotFoundError('请先运行 factor_build/04，生成 04_fusion_timeseries.xlsx')

df_impl = pd.read_excel(path_impl)
df_impl[DATE_COL] = pd.to_datetime(df_impl[DATE_COL]).dt.date
df_fusion = pd.read_excel(path_fusion)
df_fusion[DATE_COL] = pd.to_datetime(df_fusion[DATE_COL]).dt.date

# 匹配 implication 列名（可能有空格等）
y_col = 'fusion'
yy_col = next((c for c in df_impl.columns if '大盘' in str(c) and '小盘' in str(c)), None)
vol_col = next((c for c in df_impl.columns if '波动率' in str(c)), None)
bond_col = next((c for c in df_impl.columns if '股债' in str(c) or '风险平均' in str(c)), None)
if yy_col is None:
    raise KeyError('01 输出中未找到大盘涨跌-小盘涨跌列')

merged = df_impl[[DATE_COL, yy_col]].merge(df_fusion[[DATE_COL, 'fusion']], on=DATE_COL, how='inner')
if vol_col:
    merged = merged.merge(df_impl[[DATE_COL, vol_col]], on=DATE_COL, how='left')
if bond_col:
    merged = merged.merge(df_impl[[DATE_COL, bond_col]], on=DATE_COL, how='left')

x_list = [yy_col]
if vol_col and vol_col in merged.columns:
    x_list.append(vol_col)
if bond_col and bond_col in merged.columns:
    x_list.append(bond_col)

df_reg = merged[['fusion'] + x_list].dropna()
print('回归样本数:', len(df_reg))
print('因变量: fusion（融合因子）')
print('自变量:', x_list)

回归样本数: 1654
因变量: fusion（融合因子）
自变量: ['大盘涨跌-小盘涨跌', '中国波动率收盘价', '中证股债风险平均指数收盘价']


## 2. 单因子 OLS 回归（fusion 对每个自变量分别回归）

In [11]:
import statsmodels.api as sm

y = df_reg['fusion'].astype(float)
results = []
for x_col in x_list:
    df_one = df_reg[['fusion', x_col]].dropna()
    if len(df_one) < 10:
        continue
    X = sm.add_constant(df_one[x_col].astype(float))
    model = sm.OLS(df_one['fusion'].astype(float), X).fit()
    # 自变量系数（非 const）
    coef_x = model.params.get(x_col, model.params.iloc[1])
    bse_x = model.bse.get(x_col, model.bse.iloc[1])
    t_x = model.tvalues.get(x_col, model.tvalues.iloc[1])
    p_x = model.pvalues.get(x_col, model.pvalues.iloc[1])
    results.append({
        '自变量': x_col,
        'const': model.params['const'],
        '系数': coef_x,
        'std err': bse_x,
        't': t_x,
        'P>|t|': p_x,
        'R2': model.rsquared,
        'n': len(df_one),
    })
    print(f'--- fusion ~ {x_col} ---')
    print(model.summary().tables[1])
    print()

res_df = pd.DataFrame(results)
print('单因子回归汇总')
display(res_df)

--- fusion ~ 大盘涨跌-小盘涨跌 ---
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0336      0.013      2.581      0.010       0.008       0.059
大盘涨跌-小盘涨跌      0.0464      0.011      4.420      0.000       0.026       0.067

--- fusion ~ 中国波动率收盘价 ---
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0175      0.061     -0.288      0.773      -0.136       0.101
中国波动率收盘价       0.0026      0.003      0.847      0.397      -0.003       0.009

--- fusion ~ 中证股债风险平均指数收盘价 ---
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const             0.2154      0.166      1.300      0.194      -0.109       0.540
中证股债风险平均指数收盘价 -8.276e-05   7.48e-05 

,自变量,const,系数,std err,t,P>|t|,R2,n
0,大盘涨跌-小盘涨跌,0.033618,0.046440,0.010508,4.419622,0.000011,0.011686,1654
1,中国波动率收盘价,-0.017485,0.002588,0.003055,0.847225,0.396993,0.000434,1654
2,中证股债风险平均指数收盘价,0.215411,-0.000083,0.000075,-1.106577,0.268638,0.000741,1654


## 3. 输出回归结果

In [12]:
os.makedirs(FT_OUT, exist_ok=True)
out_path = os.path.join(FT_OUT, '03_regression_fusion_implication.xlsx')
with pd.ExcelWriter(out_path, engine='openpyxl') as w:
    res_df.to_excel(w, sheet_name='单因子回归汇总', index=False)
    df_reg.head(500).to_excel(w, sheet_name='样本', index=False)
print('已写出:', out_path)

已写出: /Users/zhanghongyi/Desktop/Github/VINS Project/b_program/p3_adjusted_program/factor_test/outputs/03_regression_fusion_implication.xlsx
